#**CHAPTER 1.GOVERNED CHAIN OF THOUGHTS**
---

##REFERENCE

https://chatgpt.com/share/69978397-da40-8012-af73-a2e0a38bbaab

##0.CONTEXT

**Board Briefing Memo — Deterministic Chain Reasoning Under Governance (Notebook 1/5)**

**Purpose of this paper**

This paper explains, in plain board-ready terms, what the “Reasoning AI in Finance” pipeline is doing in Notebook 1 (Chain Reasoning), what it produces, what it refuses to do, and why that matters for governance, accountability, and risk. The notebook is not a “chatbot demo.” It is a controlled, auditable workflow that behaves like a finance process: it takes a bounded input packet, applies an explicit reasoning structure, enforces separation of facts versus assumptions, detects unsupported claims, and produces a final memo plus a governance bundle that can be reviewed, archived, and audited.

The key idea is simple: in professional finance, the decision is not only “what conclusion did we reach?” The decision is also “can we explain how we reached it, what we relied on, what we assumed, what we do not know, and what must be verified before anyone acts?” This notebook is designed to make that second question answerable, every time, on every run.

**What problem are we solving**

Boards and committees frequently receive analyses that look decisive but are not defensible. The risk is not that an analyst makes a mistake—mistakes happen. The risk is that we cannot reconstruct the decision logic later, or that we cannot distinguish what was known from what was guessed, or that we discover the analysis implicitly relied on external claims that were never reviewed. In the context of AI, that risk is magnified: a model can generate plausible narratives that include invented details (“hallucinations”) unless we apply controls.

So the problem is governance, not creativity. We are building a pipeline that makes AI output behave like a reviewed finance workpaper: explicit inputs, explicit transformations, explicit gates, explicit artifacts, explicit limitations, explicit escalation when controls fail. The goal is not autonomy. The goal is **controlled inference** with **traceability**.

**What we built in Notebook 1**

Notebook 1 implements a deterministic “chain reasoning” pipeline for a synthetic deal-screening question:

**Deal question**
“Should we proceed to diligence at indicative valuation?”

**Input packet (synthetic)**
A mid-market industrial services firm with simplified financial statements, balance sheet highlights, operating KPIs, and deal context. “Synthetic” means the packet is intentionally constructed for teaching and testing controls. That is important: the notebook is not claiming to know real market comps, industry benchmarks, or external facts. It is proving that the governance scaffolding works.

**Output**
A board-facing memo in structured JSON, plus an auditable bundle of run artifacts:
- run_manifest.json (what ran, when, with what configuration)
- prompts_log.jsonl (redacted prompt and response hashes for reproducibility and audit trails)
- reasoning_trace.json (the step-by-step chain with facts used, assumptions, derived values, and uncertainties)
- risk_log.json (control failures, severity, escalation)
- final_report.json (the final memo: facts, assumptions, analysis, recommendation, open items, “Not verified”)
- deliverables.zip (packaged archive of the above)

The crucial point: the board does not need to “trust the model.” The board needs to trust the controls around the model.

**The reasoning shape: a single controlled chain**

A chain is the simplest reasoning structure: a single ordered sequence of steps. That matters because it prevents uncontrolled branching, reduces complexity, and keeps the audit trail easy to inspect. Each step is a structured record with the same required fields:

- **step_id**: unique identifier (S1, S2, …)
- **input_facts_used**: which keys from the bounded packet were referenced
- **assumptions_added**: explicit assumptions introduced at that step (if any)
- **derived_values**: outputs computed from facts/assumptions
- **uncertainty_notes**: what remains unknown or fragile
- **next_step_rationale**: why the chain proceeds to the next step

Because every step has the same schema, the chain is reviewable. A reviewer can scan: “What facts did you use?” “What assumptions did you introduce?” “What did you compute?” “Where is uncertainty acknowledged?” “What is the rationale for moving forward?”

**Pipeline overview: how the notebook runs end-to-end**

The notebook is structured into 10 executable cells. This is not arbitrary. It enforces a standard operating procedure (SOP) format:

**Cell 1 — Environment and determinism**
We install pinned dependencies and set determinism controls (fixed random seed, fixed PYTHONHASHSEED, fixed synthetic data). The purpose is reproducibility: two runs under the same configuration should generate the same synthetic packet and the same deterministic computations. If a result changes, we know why.

**Cell 2 — Governance schemas and helpers**
We define JSON schemas for the reasoning trace and for the final report, and we load validation functions. We also define hashing and redaction helpers. This is the “control layer” that ensures we can enforce structure and avoid leaking sensitive content (e.g., secrets, PII patterns) into logs.

**Cell 3 — Bounded input packet**
We generate a deterministic synthetic case and explicitly define the input boundary: “use only these figures; do not invent market comps or external facts.” This boundary is not optional; it is a governance constraint.

**Cell 4 — LLM wrapper with prompt logging**
We connect to Anthropic Claude with a fixed model and temperature 0 (deterministic behavior as much as the API allows). Every prompt is logged in redacted form with hashes, so we can later prove what we asked and what we received, without storing sensitive content. This is the equivalent of preserving “workpapers” without exposing confidential inputs.

**Cell 5 — The chain reasoning engine**
This is the core logic. It runs a six-step chain:
1) Ingest facts
2) List missing items
3) Propose assumptions (clearly labeled)
4) Compute key metrics
5) Create valuation anchors (assumption-based)
6) Draft recommendation, caveats, and questions (LLM, schema-bounded)

Importantly: most of the chain is computed deterministically in code (metrics, anchors, open items). The LLM is used for narrative synthesis only, under strict constraints and later gated.

**Cell 6 — Gates and risk escalation**
We implement three gates:
- Gate A: facts vs assumptions separation check
- Gate B: unsupported claim detector (no invented facts, especially no new numbers)
- Gate C: schema validity check (trace + final report must validate)

If a gate fails, the decision is forced to HUMAN_REVIEW, and a high-severity risk entry is written.

**Cell 7 — Trace normalization and serialization**
We write reasoning_trace.json, stripping internal objects and ensuring schema compliance.

**Cell 8 — Final report composition**
We compose final_report.json with explicit sections:
- facts_provided (verbatim from the packet)
- assumptions_introduced (explicit list)
- analysis (metrics + anchors)
- recommendation (GO / NO-GO / HUMAN_REVIEW)
- confidence level with rationale
- open_items and questions_to_verify
- verification_status = “Not verified”

The memo is board-facing: it can be read without looking at code.

**Cell 9 — Orchestration and artifact writing**
We run the pipeline, validate everything, log risks, and write artifacts.

**Cell 10 — Packaging**
We zip outputs to deliverables.zip and print a minimal path summary.

This structure is “production-clean” because it is repeatable, enforceable, and reviewable.

**What the pipeline computes (and what it refuses to do)**

The pipeline computes a small set of core finance outputs that are typical for early-stage deal screening:

**Metrics**
- Revenue
- Gross profit
- EBITDA and EBITDA margin
- EBIT
- Leverage proxy (Debt/EBITDA)
- FCF proxy (NOPAT proxy + D&A – capex), plus FCF proxy margin

These are computed from the provided synthetic statements plus explicitly stated assumptions (e.g., tax rate proxy).

**Valuation anchors**
We generate EV/EBITDA anchor values using assumed multiples (for example, 6.0x to 9.0x). These are not “market comps.” They are simply internal anchor ranges used to discuss indicative valuation sensitivity. The notebook explicitly labels them as assumptions and prohibits the model from claiming they are market-derived.

**Recommendation**
The recommendation is constrained to three outcomes:
- GO
- NO-GO
- HUMAN_REVIEW

The key governance policy is: if we detect invented facts, schema failure, or insufficient separation of facts and assumptions, we force HUMAN_REVIEW. This is the opposite of “automation bias.” The system is designed to stop and escalate rather than guess.

**What it refuses to do**
- It does not fetch market data.
- It does not quote real comparable multiples.
- It does not claim industry benchmarks.
- It does not “fill in” missing items as facts.
- It does not label any output as verified.

This refusal is a feature, not a limitation. It is how we prevent AI from becoming a liability.

**The governance gates: why they matter for the board**

**Gate A: Facts vs assumptions separation**
This gate checks that the memo includes an explicit assumptions list. Without this, any “analysis” is not controllable. In board settings, decisions depend on what is assumed. This gate enforces the discipline that assumptions are surfaced, not hidden.

**Gate B: Unsupported claim detector**
This is the critical AI risk control. The gate compares numbers present in the final output to the numbers present in:
- the input facts
- computed metrics
- valuation anchors
- explicit assumptions

If the LLM introduces new numbers that are not derived or assumed, we treat that as an invented claim and escalate. This is a conservative policy by design. It prevents “plausible but wrong” finance statements from creeping into board materials.

**Gate C: Schema validity**
This gate ensures the report and trace match their required formats. If the schema fails, we escalate. This is operationally important because structured outputs are what allow automation of review workflows, archiving, and downstream governance.

The board benefit is that we can demonstrate a consistent standard of documentation, not ad hoc memos that vary by author.

**What the results look like (conceptually)**

The tangible results are the artifacts. At a board level, focus on three:

**final_report.json**
This is the memo you read. It contains:
- **facts_provided**: the bounded packet, verbatim
- **assumptions_introduced**: explicit list of placeholders
- **analysis**: metrics + valuation anchors
- **recommendation**: GO / NO-GO / HUMAN_REVIEW, plus caveats
- **confidence**: low/medium/high with rationale
- **open_items / questions_to_verify**: what diligence must answer
- **verification_status**: “Not verified”

This makes the memo defensible: it separates knowns from unknowns.

**reasoning_trace.json**
This is the audit trail. It shows the steps, in order, with:
- what facts were used
- what assumptions were added
- what values were derived
- what uncertainties remain
- why the next step was taken
- gate outcomes

This turns “reasoning” from an invisible mental process into an inspectable artifact.

**risk_log.json**
This is the escalation record. If the model violates policy (e.g., invented facts) or the schema breaks, the risk log records:
- risk_id
- timestamp_utc
- severity
- category
- description
- control
- status (OPEN / ESCALATED)

This is board-relevant because it supports governance reporting: you can ask “how often do we escalate?” “what categories of failure occur?” “are controls working?”

**Why this is relevant to board oversight**

Boards care about decision quality, accountability, and risk. This notebook contributes to all three:

**1) Decision quality improves because uncertainty is explicit**
Most governance failures in finance come from implicit assumptions. The pipeline forces assumptions into the open and forces missing items into an explicit diligence list. This improves decision clarity even before we talk about AI.

**2) Accountability improves because the reasoning is reconstructible**
If a decision is questioned later, we can reproduce the analysis and inspect the chain. We can answer: what was the input, what did we compute, what did we assume, what did we not know, and did any gate fail?

**3) Risk decreases because the system is designed to stop**
Traditional AI demos optimize for fluency. This pipeline optimizes for safe termination. If the model invents facts, we do not proceed. We escalate to HUMAN_REVIEW. That is the correct posture for finance.

**4) Governance becomes scalable**
If we can standardize output structure (schemas), we can scale review: compliance, risk, and audit functions can sample or automatically validate memos without reading every sentence. This is how we move from artisanal analysis to institutional-grade analysis.

**What we recommend as next steps (board-level)**

Notebook 1 proves the controlled chain pattern. It is a foundation. To move toward institutional adoption, the next steps are practical:

- **Adopt standardized memo schemas** across teams so that every AI-assisted memo looks the same and is machine-checkable.
- **Expand gate coverage** beyond numeric claims to include qualitative claims (e.g., “market leader,” “high churn”) by requiring every qualitative claim to map to a provided fact or an explicit assumption.
- **Add human sign-off workflow**: the pipeline already outputs HUMAN_REVIEW; the next step is routing escalations to named reviewers and capturing sign-off metadata (who approved, when, and why).
- **Integrate into document control systems**: artifacts should be stored in controlled repositories with retention policies, like any finance workpaper.
- **Use this as the baseline pattern** for more complex reasoning shapes (tree, loop, committee, trainable evaluation), which will be covered in Notebooks 2–5.

**Closing statement**

This notebook does not ask the board to trust AI. It asks the board to trust a governed process that uses AI as one component, bounded by controls. The outputs are not “answers.” They are structured, auditable decision-support artifacts that explicitly separate facts from assumptions and declare what is not verified. In finance, that is the difference between an interesting demonstration and a defensible institutional tool.

**Verification status**
Not verified. All figures are synthetic. No external data sources were used. Any real-world use requires controlled inputs, validation, and human approval.


##1.LIBRARIES AND ENVIRONMENT

**Cell 1 — Install, environment setup, determinism, and controlled workspace**

This cell establishes the notebook as a controlled execution environment rather than an ad hoc analysis. In a board context, this is the “operating system” layer: it defines what code dependencies are used, how repeatable the run is, and where the outputs will be written. The cell pins and installs the minimal libraries needed to run the pipeline end-to-end. That matters because unpinned dependencies introduce drift: the same notebook could behave differently next week because a library updated. Pinning versions is a governance control—reproducibility is part of defensibility.

The cell then sets determinism controls: a fixed random seed and a fixed Python hash seed. These settings ensure that anything derived from randomness (in this notebook, the synthetic case generator) is repeatable. You want your governance discussion to be about decisions and controls, not about the run producing different numbers because of nondeterministic behavior. This does not mean everything becomes perfectly deterministic—external LLM APIs can still vary—but it materially reduces variability and strengthens auditability.

Next, the cell creates a clear directory structure: `artifacts/` for governance outputs and `deliverables/` for packaged distribution. This is an operational best practice with governance consequences. When every run writes to known locations with consistent filenames, reviewers can locate evidence quickly. It also enables standard retention policies and makes it straightforward to archive a run as a “workpaper bundle.”

Finally, the cell defines a utility function for UTC timestamps. This is not cosmetic. Governance requires time consistency across logs, especially when multiple systems or reviewers may compare records across time zones. The explicit UTC requirement supports audit trails: every artifact can be traced to a consistent time reference.

In short, Cell 1 is where we turn a notebook into a controlled run: pinned dependencies, deterministic knobs, consistent folders, and standardized time. Without this foundation, later controls are weaker because you cannot reliably reproduce or inspect what happened.


In [5]:
# CELL 1/10 — Install + imports + deterministic settings + directory setup
!pip -q install "anthropic==0.45.2" "jsonschema==4.23.0"

import os, json, re, uuid, hashlib, random, zipfile, platform
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, TypedDict, Literal

random.seed(7)
os.environ["PYTHONHASHSEED"] = "7"

BASE = Path(".").resolve()
ART = BASE / "artifacts"
DELIV = BASE / "deliverables"
ART.mkdir(parents=True, exist_ok=True)
DELIV.mkdir(parents=True, exist_ok=True)

def utc_now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()

print("OK:", {"utc_now": utc_now(), "python": platform.python_version(), "cwd": str(BASE)})


OK: {'utc_now': '2026-02-19T21:27:40.686253+00:00', 'python': '3.12.12', 'cwd': '/content'}


##2.CONFIGURATION

###2.1.OVERVIEW

**Cell 2 — Governance schemas, validation, redaction, hashing, and write helpers**

This cell is the governance spine. It defines the rules of the game: how outputs must be structured, how we log what happened, and how we prevent sensitive content from leaking into artifacts. For a board audience, the key point is that we are not generating free-form text and hoping it is “good enough.” We are enforcing formal schemas that make the outputs machine-checkable and reviewable.

First, we define the JSON schemas for two critical artifacts: the final board-facing report and the reasoning trace. The final report schema forces the memo to include required sections: facts provided, assumptions introduced, analysis, recommendation, confidence, open items, and “Not verified.” This ensures every run yields a comparable deliverable and prevents the model from omitting governance-critical fields. The reasoning trace schema forces each chain step to include the same metadata fields, so the reasoning process is consistent and auditable.

Second, the cell defines validation routines. Schema validation is a control gate: if the output does not match the schema, we treat that as a failure and escalate. This protects against brittle or malformed outputs and provides a hard signal that downstream automation should not trust the artifact.

Third, the hashing utilities (SHA-256 of text and JSON) create integrity fingerprints. These fingerprints allow you to prove that an artifact corresponds to a specific run configuration and content, and to detect tampering or accidental changes. In regulated or high-stakes environments, this kind of integrity evidence is part of mature governance.

Fourth, the redaction logic is crucial. We log prompts for auditability, but we do not want to leak secrets or potential PII into persistent logs. Redaction patterns remove common PII-like structures (emails, phone numbers, IDs) and explicitly block API key leakage. The governance posture is “minimum necessary data,” even in internal artifacts.

Fifth, we define two standard file writers: `write_json` for atomic JSON writes and `append_jsonl` for line-delimited logging. Atomic writing reduces the risk of partial files if the run is interrupted. JSONL logging allows incremental prompt logs and event logs that can be streamed and audited.

Cell 2 is where we formalize the difference between a demo and an institutional tool: we define what “valid” looks like, we validate it, we log safely, and we preserve integrity evidence.


###2.2.CODE AND IMPLEMENTATION

In [6]:
# CELL 2/10 — Config + schemas + helpers (hashing, redaction, JSON writing, validation)
from jsonschema import validate, Draft202012Validator

CONFIG: Dict[str, Any] = {
    "project": "Reasoning AI in Finance — Structured Inference Under Governance",
    "notebook": "Notebook 1 — Chain Reasoning",
    "model": "claude-haiku-4-5-20251001",
    "max_prompt_chars_logged": 2000,
    "max_steps": 6,
    "unsupported_claim": {
        "max_new_numbers_allowed": 0,   # strict: any new numeric claim => HUMAN_REVIEW (unless derived/assumption)
    },
    "decision_thresholds": {
        "leverage_proxy_high": 4.0,     # Debt/EBITDA proxy
        "ebitda_margin_low": 0.10,      # 10%
        "fcf_proxy_low": 0.02,          # 2% of revenue
    },
}

FINAL_REPORT_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "run_id","timestamp_utc","project","notebook",
        "executive_summary","facts_provided","assumptions_introduced",
        "analysis","recommendation","confidence","open_items","questions_to_verify",
        "verification_status"
    ],
    "properties": {
        "run_id": {"type":"string"},
        "timestamp_utc": {"type":"string"},
        "project": {"type":"string"},
        "notebook": {"type":"string"},
        "executive_summary": {"type":"string"},
        "facts_provided": {"type":"object"},
        "assumptions_introduced": {"type":"array", "items":{"type":"string"}},
        "analysis": {
            "type":"object",
            "additionalProperties": True,
            "required": ["metrics","valuation_anchors"],
            "properties": {
                "metrics":{"type":"object"},
                "valuation_anchors":{"type":"object"},
            }
        },
        "recommendation": {"type":"object", "additionalProperties": False,
            "required":["decision","rationale","caveats"],
            "properties":{
                "decision":{"type":"string", "enum":["GO","NO-GO","HUMAN_REVIEW"]},
                "rationale":{"type":"string"},
                "caveats":{"type":"array","items":{"type":"string"}}
            }
        },
        "confidence": {"type":"object", "additionalProperties": False,
            "required":["level","rationale"],
            "properties":{
                "level":{"type":"string","enum":["low","medium","high"]},
                "rationale":{"type":"string"}
            }
        },
        "open_items": {"type":"array","items":{"type":"string"}},
        "questions_to_verify": {"type":"array","items":{"type":"string"}},
        "verification_status": {"type":"string", "enum":["Not verified"]},
    }
}

REASONING_TRACE_SCHEMA: Dict[str, Any] = {
    "type":"object",
    "additionalProperties": False,
    "required":["run_id","timestamp_utc","reasoning_shape","steps","gates","termination_reason"],
    "properties":{
        "run_id":{"type":"string"},
        "timestamp_utc":{"type":"string"},
        "reasoning_shape":{"type":"string","enum":["chain"]},
        "steps":{
            "type":"array",
            "items":{
                "type":"object",
                "additionalProperties": False,
                "required":[
                    "step_id","name","input_facts_used","assumptions_added","derived_values",
                    "uncertainty_notes","next_step_rationale"
                ],
                "properties":{
                    "step_id":{"type":"string"},
                    "name":{"type":"string"},
                    "input_facts_used":{"type":"array","items":{"type":"string"}},
                    "assumptions_added":{"type":"array","items":{"type":"string"}},
                    "derived_values":{"type":"object"},
                    "uncertainty_notes":{"type":"array","items":{"type":"string"}},
                    "next_step_rationale":{"type":"string"}
                }
            }
        },
        "gates":{
            "type":"object",
            "additionalProperties": False,
            "required":["GateA_FactsVsAssumptions","GateB_UnsupportedClaims","GateC_SchemaValidity"],
            "properties":{
                "GateA_FactsVsAssumptions":{"type":"object"},
                "GateB_UnsupportedClaims":{"type":"object"},
                "GateC_SchemaValidity":{"type":"object"},
            }
        },
        "termination_reason":{"type":"string"},
    }
}

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_json(obj: Any) -> str:
    return sha256_text(json.dumps(obj, sort_keys=True, ensure_ascii=False))

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def append_jsonl(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, sort_keys=True, ensure_ascii=False) + "\n")

PII_PATTERNS = [
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"), "[REDACTED_SSN]"),
    (re.compile(r"\b\d{10,16}\b"), "[REDACTED_ID]"),
    (re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.I), "[REDACTED_EMAIL]"),
    (re.compile(r"\b(\+?\d[\d \-().]{8,}\d)\b"), "[REDACTED_PHONE]"),
]

def redact_text(s: str) -> str:
    out = s
    for rx, repl in PII_PATTERNS:
        out = rx.sub(repl, out)
    out = re.sub(r"(?i)anthropic[_-]?api[_-]?key\s*[:=]\s*[A-Za-z0-9_\-]{10,}", "ANTHROPIC_API_KEY=[REDACTED]", out)
    return out

def validate_or_raise(schema: Dict[str, Any], obj: Any) -> None:
    Draft202012Validator.check_schema(schema)
    validate(instance=obj, schema=schema)

print("OK:", {"schemas_loaded": True, "config_model": CONFIG["model"]})


OK: {'schemas_loaded': True, 'config_model': 'claude-haiku-4-5-20251001'}


##3.SYNTHETIC FINANCE GENERATOR

###3.1.OVERVIEW

**Cell 3 — Synthetic finance case generation and explicit input boundary**

Cell 3 provides the bounded input packet that the rest of the pipeline is allowed to use. This is a governance-first design choice: we explicitly define what the system knows, and we explicitly prevent it from “knowing” anything else. In board terms, this is the equivalent of saying: “Here is the data room excerpt we are using, and we are not going beyond it.”

The case is synthetic and deterministic. “Synthetic” means the numbers are not claimed to be real-world market facts; they exist to test and demonstrate the reasoning and governance controls. “Deterministic” means the same seed generates the same packet every time. This is valuable for teaching, testing, and auditing because you can reproduce the run precisely and compare outputs over time.

The case includes simplified income statement figures (revenue, costs, operating expenses, D&A, interest), balance sheet highlights (cash, debt, working capital proxy), cash flow highlights (capex), operating KPIs, and basic deal context. Importantly, it also includes an explicit “input boundary notice” that tells the model: do not introduce external market comps, do not fabricate industry benchmarks, and do not claim facts that are not present. This boundary notice is not merely instructional; it is part of the governance evidence that the system is intended to operate within constraints.

In addition, the case includes a curated list of open items. This is deliberately designed: the pipeline should not pretend the packet is complete. Instead, it should surface missing diligence items explicitly (customer concentration, covenant details, working capital seasonality, QoE adjustments policy, and so on). In real finance processes, these are exactly the questions that determine whether an indicative view is meaningful.

Cell 3 therefore accomplishes three essential things: it creates a consistent testbed, it defines the universe of permissible facts, and it seeds the diligence mindset. Without a bounded packet, an LLM can “fill in the gaps” with plausible inventions. By defining the boundary here, we set the stage for later gates to detect and stop unsupported claims.


###3.2.CODE AND IMPLEMENTATION

In [7]:
# CELL 3/10 — Synthetic finance case generator (deterministic) + input boundary definition
class SyntheticCase(TypedDict):
    company: str
    deal_question: str
    facts_provided: Dict[str, Any]
    open_items_seed: List[str]

def generate_synthetic_case(seed: int = 7) -> SyntheticCase:
    rng = random.Random(seed)
    company = "MidCo Industrial Services, Inc. (synthetic)"
    deal_question = "Should we proceed to diligence at indicative valuation?"
    revenue = 420_000_000  # USD
    cogs = 286_000_000
    opex = 86_000_000
    d_and_a = 18_000_000
    interest = 22_000_000
    cash = 24_000_000
    total_debt = 210_000_000
    net_working_capital = 52_000_000
    capex = 20_000_000
    tax_rate_assumption = 0.25  # clearly an assumption seed; but included in case as "stated policy proxy"
    kpis = {
        "sites_served": 145,
        "utilization_rate": 0.79,
        "customer_retention_rate": 0.92,
        "on_time_service_rate": 0.95,
    }
    facts = {
        "period": "LTM (synthetic)",
        "currency": "USD",
        "income_statement": {
            "revenue": revenue,
            "cogs": cogs,
            "opex": opex,
            "d_and_a": d_and_a,
            "interest_expense": interest,
        },
        "balance_sheet_highlights": {
            "cash": cash,
            "total_debt": total_debt,
            "net_working_capital": net_working_capital,
        },
        "cash_flow_highlights": {
            "capex": capex,
        },
        "operating_kpis": kpis,
        "deal_context": {
            "seller_type": "private equity (synthetic)",
            "process": "limited auction (synthetic)",
            "proposed_structure": "asset purchase (synthetic)",
        },
        "input_boundary_notice": (
            "This packet is synthetic. Use only provided figures for facts. "
            "Do not introduce market comps or external facts."
        ),
    }
    open_items_seed = [
        "Customer concentration (top-10 share) and contract terms",
        "Churn and pricing escalators by customer cohort",
        "Maintenance vs growth capex split",
        "Working capital seasonality and normalization bridge",
        "One-time costs and non-recurring EBITDA adjustments policy",
        "Debt covenant terms, maturities, and interest rate structure",
        "Backlog visibility and pipeline conversion",
    ]
    return {
        "company": company,
        "deal_question": deal_question,
        "facts_provided": facts,
        "open_items_seed": open_items_seed,
    }

CASE = generate_synthetic_case(7)
print("SYNTHETIC CASE READY:", CASE["company"])
print("FACT KEYS:", list(CASE["facts_provided"].keys()))


SYNTHETIC CASE READY: MidCo Industrial Services, Inc. (synthetic)
FACT KEYS: ['period', 'currency', 'income_statement', 'balance_sheet_highlights', 'cash_flow_highlights', 'operating_kpis', 'deal_context', 'input_boundary_notice']


##4.LLM CLIENT WRAPPER

###4.1.OVERVIEW

**Cell 4 — Anthropic LLM wrapper with strict logging, redaction, and JSON parsing discipline**

Cell 4 is where the notebook connects to the external model while maintaining governance controls. From a board perspective, this cell demonstrates a safe integration pattern: we use an LLM as a component, but we wrap it with logging, redaction, hashing, and strict parsing so that the model’s output cannot silently become “truth.”

First, the cell retrieves the API key from Colab Secrets and refuses to run if the key is missing. This is a security best practice: secrets should not be hard-coded or written into notebooks or artifacts. The pipeline never logs the raw key, and the redaction routines explicitly guard against accidental key leakage.

Second, the wrapper fixes the model name and uses temperature 0 to reduce stochastic variability. This is not about claiming perfect determinism, but about reducing randomness to support repeatability and review.

Third, the wrapper logs prompts and responses in a governance-compatible way. It does not store raw prompts indiscriminately. It stores redacted prompt text (truncated) plus cryptographic hashes of the prompt and of a “schema hint.” This gives auditors a chain of evidence: we can prove what was asked (in redacted form), what schema constraints were intended, and what response was returned (via hash), without persisting sensitive or excessive content.

Fourth, the wrapper enforces a strict expectation: the LLM must return valid JSON. If the response cannot be parsed as JSON, the wrapper returns a structured error marker. This matters because “non-parseable output” is not a minor inconvenience in governed systems; it is a control failure. Downstream gates can detect this and force HUMAN_REVIEW.

Finally, the wrapper’s design supports separation of concerns: the reasoning engine can request a JSON output with a specified structure, and the wrapper ensures we either get valid JSON or a clearly flagged failure. That reduces the chance of hidden errors and makes the pipeline behavior inspectable.

Cell 4 is therefore the “safe interface” between governed finance processes and an inherently generative model. The board takeaway is that we are not letting the model speak directly into decision memos without mediation, logging, and enforceable structure.


###4.2.CODE AND IMPLEMENTATION

In [9]:
# CELL 4/10 — LLM client wrapper (Anthropic) + prompt logging (redacted + hashes)
from google.colab import userdata
from anthropic import Anthropic

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY or not isinstance(ANTHROPIC_API_KEY, str):
    raise RuntimeError("Missing ANTHROPIC_API_KEY in Colab Secrets.")

client = Anthropic(api_key=ANTHROPIC_API_KEY)

PROMPTS_LOG = ART / "prompts_log.jsonl"

def llm_call_json(system: str, user: str, schema_hint: Dict[str, Any], run_id: str, call_id: str) -> Dict[str, Any]:
    # Log prompt (redacted) + hashes; never log raw key
    ts = utc_now()
    redacted_system = redact_text(system)[:CONFIG["max_prompt_chars_logged"]]
    redacted_user = redact_text(user)[:CONFIG["max_prompt_chars_logged"]]
    prompt_fingerprint = sha256_text(redacted_system + "\n" + redacted_user)

    append_jsonl(PROMPTS_LOG, {
        "run_id": run_id,
        "call_id": call_id,
        "timestamp_utc": ts,
        "model": CONFIG["model"],
        "system_redacted": redacted_system,
        "user_redacted": redacted_user,
        "schema_hint_hash": sha256_json(schema_hint),
        "prompt_hash": prompt_fingerprint,
    })

    msg = client.messages.create(
        model=CONFIG["model"],
        max_tokens=900,
        temperature=0,
        system=system,
        messages=[{"role":"user","content":user}],
    )

    text = ""
    for block in msg.content:
        if getattr(block, "type", None) == "text":
            text += block.text

    # Attempt strict JSON parse; if fails, return an error wrapper (handled by gates)
    parsed: Dict[str, Any]
    try:
        parsed = json.loads(text)
        parse_ok = True
    except Exception:
        parsed = {"_parse_error": True, "_raw_text": text[:5000]}
        parse_ok = False

    append_jsonl(PROMPTS_LOG, {
        "run_id": run_id,
        "call_id": call_id,
        "timestamp_utc": utc_now(),
        "response_parse_ok": parse_ok,
        "response_hash": sha256_text(text),
    })
    return parsed

print("LLM CLIENT READY:", {"model": CONFIG["model"], "prompts_log": str(PROMPTS_LOG)})


LLM CLIENT READY: {'model': 'claude-haiku-4-5-20251001', 'prompts_log': '/content/artifacts/prompts_log.jsonl'}


##5.REASONING ENGINE CORE

###5.1.OVERVIEW

**Cell 5 — The chain reasoning engine: single-path logic, deterministic computations, and constrained LLM synthesis**

Cell 5 is the heart of Notebook 1. It implements the deterministic chain reasoning pattern as a sequence of explicit steps that mirror a disciplined analyst workflow. The main function, `run_chain(case)`, produces four things: a reasoning trace, draft memo sections, a risk list, and a preliminary decision placeholder. This design makes the reasoning process a first-class output, not a hidden internal behavior.

The chain is intentionally single-path. There is no branching, no iteration, and no committee. That simplicity is a governance advantage at this stage: it keeps the logic reviewable and reduces the surface area for failure. The chain steps follow a finance-credible progression: ingest bounded facts, list missing items, state assumptions, compute key metrics, form valuation anchors, and draft a recommendation with caveats and open questions.

A critical design choice is that the metric computations and valuation anchors are computed deterministically in Python, not “invented” by the LLM. EBITDA, margins, leverage proxies, and FCF proxies come from the provided packet plus explicitly stated assumptions (like a tax proxy). Valuation anchors are produced using assumed multiple ranges and are labeled as assumptions rather than market facts. This separation ensures that quantitative results are transparent and reproducible.

The LLM is used only for narrative synthesis in the final step: executive summary, rationale, caveats, questions to verify, and confidence language. Even here, it is tightly constrained: the system message instructs the model to use only the provided facts and stated assumptions, not to introduce new numbers, and to return only JSON matching a schema hint. The wrapper logs prompts and enforces JSON parsing, which later gates can validate.

Within the trace, each step records the keys of facts used, assumptions added, derived values, uncertainty notes, and the rationale for proceeding. This yields a reasoning_trace artifact that is comparable to a workpaper trail.

In pedagogical terms, Cell 5 demonstrates how to “engineer reasoning” as a structured pipeline rather than a conversational output. The board should see that the notebook turns analyst judgment into documented steps, without allowing the model to smuggle in unsupported claims.


###5.2.CODE AND IMPLEMENTATION

In [10]:
# CELL 5/10 — Reasoning engine core (CHAIN): run_chain(case) -> (trace, draft_sections, risks, decision)
class Risk(TypedDict):
    risk_id: str
    timestamp_utc: str
    severity: Literal["low","medium","high"]
    category: str
    description: str
    control: str
    status: Literal["OPEN","MITIGATED","ESCALATED"]

class ChainStep(TypedDict):
    step_id: str
    name: str
    input_facts_used: List[str]
    assumptions_added: List[str]
    derived_values: Dict[str, Any]
    uncertainty_notes: List[str]
    next_step_rationale: str

class ChainDraft(TypedDict, total=False):
    executive_summary: str
    rationale: str
    caveats: List[str]
    questions_to_verify: List[str]
    confidence_level: Literal["low","medium","high"]
    confidence_rationale: str

def _get_fact(case: SyntheticCase, path: List[str]) -> Any:
    cur: Any = case["facts_provided"]
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return None
        cur = cur[p]
    return cur

def _compute_metrics(case: SyntheticCase, assumptions: Dict[str, Any]) -> Dict[str, Any]:
    is_ = case["facts_provided"]["income_statement"]
    bs_ = case["facts_provided"]["balance_sheet_highlights"]
    cf_ = case["facts_provided"]["cash_flow_highlights"]

    revenue = float(is_["revenue"])
    cogs = float(is_["cogs"])
    opex = float(is_["opex"])
    d_and_a = float(is_["d_and_a"])
    interest = float(is_["interest_expense"])
    total_debt = float(bs_["total_debt"])
    capex = float(cf_["capex"])

    gross_profit = revenue - cogs
    ebitda = revenue - cogs - opex
    ebit = ebitda - d_and_a
    ebitda_margin = ebitda / revenue if revenue else 0.0

    leverage_proxy = (total_debt / ebitda) if ebitda else None

    tax_rate = float(assumptions.get("tax_rate", 0.25))
    nopat_proxy = ebit * (1.0 - tax_rate)
    fcf_proxy = nopat_proxy + d_and_a - capex
    fcf_proxy_margin = (fcf_proxy / revenue) if revenue else 0.0

    return {
        "revenue": revenue,
        "gross_profit": gross_profit,
        "ebitda": ebitda,
        "ebit": ebit,
        "ebitda_margin": ebitda_margin,
        "leverage_proxy_debt_to_ebitda": leverage_proxy,
        "fcf_proxy": fcf_proxy,
        "fcf_proxy_margin": fcf_proxy_margin,
        "interest_expense": interest,
        "capex": capex,
        "tax_rate_assumption_used": tax_rate,
    }

def _valuation_anchors(metrics: Dict[str, Any], assumptions: Dict[str, Any]) -> Dict[str, Any]:
    ebitda = float(metrics["ebitda"])
    if ebitda <= 0:
        return {"note": "EBITDA <= 0; indicative multiples not meaningful under current packet."}
    # Assumptions (NOT market facts): simple ranges as internal anchors
    low_mult = float(assumptions.get("ev_ebitda_low", 6.0))
    mid_mult = float(assumptions.get("ev_ebitda_mid", 7.5))
    high_mult = float(assumptions.get("ev_ebitda_high", 9.0))
    ev_low = ebitda * low_mult
    ev_mid = ebitda * mid_mult
    ev_high = ebitda * high_mult
    return {
        "method": "EV/EBITDA (assumption-based anchors; not market comps)",
        "multiples_assumed": {"low": low_mult, "mid": mid_mult, "high": high_mult},
        "enterprise_value_range": {"low": ev_low, "mid": ev_mid, "high": ev_high},
    }

def run_chain(case: SyntheticCase) -> Tuple[Dict[str, Any], ChainDraft, List[Risk], str]:
    risks: List[Risk] = []
    steps: List[ChainStep] = []
    assumptions_list: List[str] = []
    assumptions_kv: Dict[str, Any] = {}

    def add_risk(sev: Literal["low","medium","high"], cat: str, desc: str, control: str, status: Literal["OPEN","MITIGATED","ESCALATED"]="OPEN") -> None:
        risks.append({
            "risk_id": "R-" + uuid.uuid4().hex[:8],
            "timestamp_utc": utc_now(),
            "severity": sev,
            "category": cat,
            "description": desc,
            "control": control,
            "status": status,
        })

    # Step 1: ingest facts (bounded)
    steps.append({
        "step_id": "S1",
        "name": "Ingest bounded facts",
        "input_facts_used": ["income_statement","balance_sheet_highlights","cash_flow_highlights","operating_kpis","deal_context","input_boundary_notice"],
        "assumptions_added": [],
        "derived_values": {"facts_ingested": True},
        "uncertainty_notes": ["Packet is synthetic and incomplete by design."],
        "next_step_rationale": "Confirm what is known before listing unknowns and assumptions.",
    })

    # Step 2: list missing items (deterministic from seed list)
    open_items = list(case["open_items_seed"])
    steps.append({
        "step_id": "S2",
        "name": "List missing items that drive the decision",
        "input_facts_used": ["open_items_seed"],
        "assumptions_added": [],
        "derived_values": {"missing_items": open_items},
        "uncertainty_notes": ["Missing items could materially change valuation and go/no-go."],
        "next_step_rationale": "Propose explicit assumptions only as placeholders; never treat them as facts.",
    })

    # Step 3: propose assumptions (clearly labeled; deterministic defaults)
    assumptions_kv = {
        "tax_rate": 0.25,
        "ev_ebitda_low": 6.0,
        "ev_ebitda_mid": 7.5,
        "ev_ebitda_high": 9.0,
        "wc_normalization_delta": 0.0,
        "non_recurring_adjustments": 0.0,
    }
    assumptions_list = [
        "Tax rate proxy set to 25% for NOPAT/FCF proxy (assumption).",
        "EV/EBITDA anchor range assumed at 6.0x–9.0x (assumption; not market comps).",
        "Working capital normalization delta assumed 0 (assumption pending seasonality data).",
        "Non-recurring EBITDA adjustments assumed 0 (assumption pending QoE policy).",
    ]
    steps.append({
        "step_id": "S3",
        "name": "Introduce placeholder assumptions (explicitly labeled)",
        "input_facts_used": ["input_boundary_notice"],
        "assumptions_added": assumptions_list,
        "derived_values": {"assumptions_kv": assumptions_kv},
        "uncertainty_notes": ["Assumptions are placeholders; require verification in diligence."],
        "next_step_rationale": "Compute key metrics using only facts + stated assumptions.",
    })

    # Step 4: compute key metrics
    metrics = _compute_metrics(case, assumptions_kv)
    steps.append({
        "step_id": "S4",
        "name": "Compute key metrics (facts + stated assumptions only)",
        "input_facts_used": ["income_statement.revenue","income_statement.cogs","income_statement.opex","income_statement.d_and_a",
                            "income_statement.interest_expense","balance_sheet_highlights.total_debt","cash_flow_highlights.capex"],
        "assumptions_added": ["Tax rate proxy (for NOPAT/FCF proxy)."],
        "derived_values": {"metrics": metrics},
        "uncertainty_notes": ["FCF is a proxy; WC movements and one-time items are unknown."],
        "next_step_rationale": "Translate metrics into indicative valuation anchors without external comps.",
    })

    # Step 5: valuation anchors
    anchors = _valuation_anchors(metrics, assumptions_kv)
    steps.append({
        "step_id": "S5",
        "name": "Create indicative valuation anchors (assumption-based)",
        "input_facts_used": ["metrics.ebitda"],
        "assumptions_added": ["EV/EBITDA multiple range (assumption)."],
        "derived_values": {"valuation_anchors": anchors},
        "uncertainty_notes": ["Anchors are not market-based; only internal range-setting for diligence decision."],
        "next_step_rationale": "Draft board-facing recommendation with caveats and open questions.",
    })

    # Step 6: draft recommendation via LLM (bounded, schema-driven)
    draft_schema_hint = {
        "type":"object",
        "additionalProperties": False,
        "required": ["executive_summary","rationale","caveats","questions_to_verify","confidence_level","confidence_rationale"],
        "properties":{
            "executive_summary":{"type":"string"},
            "rationale":{"type":"string"},
            "caveats":{"type":"array","items":{"type":"string"}},
            "questions_to_verify":{"type":"array","items":{"type":"string"}},
            "confidence_level":{"type":"string","enum":["low","medium","high"]},
            "confidence_rationale":{"type":"string"},
        }
    }

    system = (
        "You are a finance analyst writing a board-ready memo. "
        "STRICT GOVERNANCE: Use ONLY the provided facts and explicitly stated assumptions. "
        "Do NOT introduce any external market data, comps, industry facts, or new numbers. "
        "If something is missing, put it as a question_to_verify. "
        "Return ONLY valid JSON matching the provided schema."
    )
    user = json.dumps({
        "company": case["company"],
        "deal_question": case["deal_question"],
        "facts_provided": case["facts_provided"],
        "assumptions_introduced": assumptions_list,
        "derived": {"metrics": metrics, "valuation_anchors": anchors},
        "required_output_schema": draft_schema_hint,
    }, ensure_ascii=False)

    run_id = "__RUN_ID_PLACEHOLDER__"  # overwritten by orchestrator via string replace pattern
    draft = llm_call_json(system=system, user=user, schema_hint=draft_schema_hint, run_id=run_id, call_id="LLM_DRAFT")

    steps.append({
        "step_id": "S6",
        "name": "Draft recommendation + caveats + open questions (LLM; schema-bounded)",
        "input_facts_used": ["facts_provided","assumptions_introduced","derived.metrics","derived.valuation_anchors"],
        "assumptions_added": [],
        "derived_values": {"draft_keys": list(draft.keys())},
        "uncertainty_notes": ["LLM output must be checked for invented facts and schema validity."],
        "next_step_rationale": "Run governance gates before finalizing report.",
    })

    trace = {
        "run_id": run_id,
        "timestamp_utc": utc_now(),
        "reasoning_shape": "chain",
        "steps": steps,
        "gates": {
            "GateA_FactsVsAssumptions": {},
            "GateB_UnsupportedClaims": {},
            "GateC_SchemaValidity": {},
        },
        "termination_reason": "COMPLETED_CHAIN",
        "_internal": {
            "assumptions_list": assumptions_list,
            "assumptions_kv": assumptions_kv,
            "metrics": metrics,
            "valuation_anchors": anchors,
            "open_items": open_items,
            "draft": draft,
        }
    }

    # Decision placeholder (set after gates)
    decision = "HUMAN_REVIEW"
    return trace, draft, risks, decision

print("CHAIN ENGINE READY:", {"max_steps": CONFIG["max_steps"]})


CHAIN ENGINE READY: {'max_steps': 6}


##6.RISK DETECTION

###6.1.OVERVIEW

**Cell 6 — Governance gates: separation check, unsupported claim detection, and schema enforcement with escalation**

Cell 6 is where the notebook becomes governance-grade. The chain engine can produce a draft, but we do not accept it blindly. We run gates that explicitly test whether the output meets professional standards: facts and assumptions must be separable, invented claims must be blocked, and schemas must validate. If any gate fails, we log a high-severity risk and force HUMAN_REVIEW.

Gate A checks that assumptions are present and explicitly listed. This is a minimal but critical control. Without an assumptions list, a memo can appear precise while embedding unreviewed placeholders. A board does not approve “analysis”; it approves analysis conditional on assumptions being reasonable. Gate A ensures the memo can be reviewed on that basis.

Gate B is the core AI safety control in this notebook. It is designed to detect unsupported numeric claims. The logic extracts all numbers appearing in the final narrative draft and compares them to the numbers that legitimately exist in (a) the bounded facts, (b) computed metrics, (c) valuation anchors, and (d) explicit assumptions. If the model outputs a new number that is not present in these sources, it is treated as a potential invented fact and triggers escalation. The policy is conservative on purpose: in finance, “one wrong number” can dominate an interpretation. Even if the model is usually correct, the governance requirement is to catch the cases when it is not.

Gate B also treats a JSON parse error as a failure. If the output is not parseable, it cannot be considered a controlled artifact and must escalate.

Gate C validates the schemas for both the reasoning trace and the final report. This ensures that downstream users (humans and systems) can rely on the structure being stable. It also prevents partial or malformed outputs from being quietly used.

The risk log entries include standardized fields: risk_id, timestamp, severity, category, description, control, and status. This is important for board oversight because it enables aggregation: how often do we fail a gate, why, and what is the remediation.

Cell 6 therefore demonstrates the governance philosophy of the entire project: **capability must be bounded by controls, and failures must be explicit and escalated, not hidden.**


###6.2.CODE AND IMPLEMENTATION

In [11]:
# CELL 6/10 — Gate(s) + risk detection + escalation logic
NUM_RX = re.compile(r"(?<![A-Za-z])(-?\d{1,3}(?:,\d{3})*(?:\.\d+)?%?)(?![A-Za-z])")

def extract_numbers(obj: Any) -> List[str]:
    txt = json.dumps(obj, ensure_ascii=False)
    return NUM_RX.findall(txt)

def gate_a_facts_vs_assumptions(trace: Dict[str, Any], risks: List[Risk]) -> Dict[str, Any]:
    internal = trace.get("_internal", {})
    facts = internal.get("case_facts")  # may be absent
    assumptions_list = internal.get("assumptions_list", [])
    ok = isinstance(assumptions_list, list) and len(assumptions_list) > 0
    if not ok:
        risks.append({
            "risk_id": "R-" + uuid.uuid4().hex[:8],
            "timestamp_utc": utc_now(),
            "severity": "high",
            "category": "GOVERNANCE",
            "description": "Gate A failed: assumptions_introduced missing or empty; facts vs assumptions separation not demonstrable.",
            "control": "Require explicit assumptions list; otherwise escalate to HUMAN_REVIEW.",
            "status": "ESCALATED",
        })
    return {"passed": bool(ok), "notes": "Assumptions list present and explicitly labeled." if ok else "Missing assumptions list."}

def gate_b_unsupported_claims(case: SyntheticCase, trace: Dict[str, Any], risks: List[Risk]) -> Dict[str, Any]:
    internal = trace.get("_internal", {})
    draft = internal.get("draft", {})
    metrics = internal.get("metrics", {})
    anchors = internal.get("valuation_anchors", {})
    assumptions_list = internal.get("assumptions_list", [])
    facts = case["facts_provided"]

    # Collect allowed numbers from facts + derived + assumptions (as strings)
    allowed_sources = {"facts": facts, "metrics": metrics, "anchors": anchors, "assumptions": assumptions_list}
    allowed_nums = set(extract_numbers(allowed_sources))
    draft_nums = extract_numbers(draft)

    new_nums = [n for n in draft_nums if n not in allowed_nums]
    # Also block if draft contains parse_error marker
    parse_error = bool(draft.get("_parse_error"))

    passed = (not parse_error) and (len(new_nums) <= CONFIG["unsupported_claim"]["max_new_numbers_allowed"])
    if not passed:
        desc = "Gate B failed: unsupported numeric claims or JSON parse error detected."
        if parse_error:
            desc += " LLM output was not valid JSON."
        if new_nums:
            desc += f" New numbers found: {sorted(set(new_nums))[:12]}"
        risks.append({
            "risk_id": "R-" + uuid.uuid4().hex[:8],
            "timestamp_utc": utc_now(),
            "severity": "high",
            "category": "INVENTED_FACTS",
            "description": desc,
            "control": "Block promotion; require HUMAN_REVIEW and re-run with tightened prompt or manual edits.",
            "status": "ESCALATED",
        })
    return {
        "passed": bool(passed),
        "parse_error": parse_error,
        "new_numbers_detected": sorted(set(new_nums))[:50],
        "policy": "No new numeric claims beyond facts/derived/assumptions.",
    }

def gate_c_schema_validity(final_report: Dict[str, Any], trace: Dict[str, Any], risks: List[Risk]) -> Dict[str, Any]:
    ok_trace = True
    ok_report = True
    trace_err = ""
    report_err = ""
    try:
        t2 = {k:v for k,v in trace.items() if k != "_internal"}  # strip internal
        validate_or_raise(REASONING_TRACE_SCHEMA, t2)
    except Exception as e:
        ok_trace = False
        trace_err = str(e)[:600]
    try:
        validate_or_raise(FINAL_REPORT_SCHEMA, final_report)
    except Exception as e:
        ok_report = False
        report_err = str(e)[:600]

    passed = ok_trace and ok_report
    if not passed:
        risks.append({
            "risk_id": "R-" + uuid.uuid4().hex[:8],
            "timestamp_utc": utc_now(),
            "severity": "high",
            "category": "SCHEMA",
            "description": f"Gate C failed: schema invalid. trace_ok={ok_trace}, report_ok={ok_report}.",
            "control": "Force HUMAN_REVIEW; block downstream automation until schemas validate.",
            "status": "ESCALATED",
        })
    return {
        "passed": bool(passed),
        "trace_schema_ok": ok_trace,
        "final_report_schema_ok": ok_report,
        "trace_error": trace_err,
        "final_report_error": report_err,
    }

print("GATES READY")


GATES READY


##7.TRACE BUILDER

###7.1.OVERVIEW

**Cell 7 — Trace normalization and creation of reasoning_trace.json as an auditable workpaper**

Cell 7 takes the internal representation of the chain and converts it into a clean, auditable artifact: `reasoning_trace.json`. The purpose is to ensure that the trace is not a developer’s internal object but a standardized governance deliverable. Boards do not want to read code; they want evidence. The trace file is that evidence.

The cell does two important things. First, it normalizes the trace: it removes internal helper fields that are not meant for external review and ensures required fields are present. This is governance hygiene. Internal objects may include transient state, debug data, or implementation details that could confuse reviewers or leak unnecessary information. By stripping `_internal` and enforcing a clean schema, we produce a stable workpaper format.

Second, it writes the trace to disk using the standard writer. This turns the reasoning chain into a persistent artifact that can be archived, compared across runs, and inspected independently of the notebook. The same way a finance team stores calculation worksheets and support schedules, we store the reasoning schedule.

The normalized trace includes the reasoning shape (“chain”), the ordered list of steps with their required fields, the gate outcomes, and a termination reason. This is critical: the trace is not just “what we did,” but also “whether controls passed.” A reviewer can immediately see whether the pipeline’s outputs are safe to consider or whether escalation occurred.

From a pedagogical standpoint, Cell 7 illustrates a design principle: **reasoning must be serialized**. If you cannot persist the reasoning in a structured form, you cannot audit it, automate review around it, or confidently reuse it. This is the difference between a model producing a persuasive paragraph and a governed system producing a traceable decision support artifact.

In board terms, the reasoning trace is the mechanism that allows accountability. It supports questions like: “What did we rely on?” “Where did assumptions enter?” “Did we have any gate failures?” “What uncertainties were flagged?” If these questions can be answered from a single file, the organization can treat AI-assisted memos as institutional workpapers rather than ephemeral chat outputs.


###7.2.CODE AND IMPLEMENTATION

In [12]:
# CELL 7/10 — Trace builder (reasoning_trace.json) + normalization
def normalize_trace_for_write(trace: Dict[str, Any], run_id: str, gates: Dict[str, Any]) -> Dict[str, Any]:
    out = {k:v for k,v in trace.items() if k != "_internal"}  # drop internal
    out["run_id"] = run_id
    out["timestamp_utc"] = out.get("timestamp_utc") or utc_now()
    out["gates"] = gates
    # Ensure step order and ids
    steps = out.get("steps", [])
    if isinstance(steps, list):
        for i, s in enumerate(steps):
            if "step_id" not in s:
                s["step_id"] = f"S{i+1}"
    out["termination_reason"] = out.get("termination_reason") or "COMPLETED_CHAIN"
    return out

def build_reasoning_trace_file(trace: Dict[str, Any], run_id: str, gates: Dict[str, Any]) -> Dict[str, Any]:
    normalized = normalize_trace_for_write(trace, run_id, gates)
    write_json(ART / "reasoning_trace.json", normalized)
    return normalized

print("TRACE BUILDER READY:", str(ART / "reasoning_trace.json"))


TRACE BUILDER READY: /content/artifacts/reasoning_trace.json


##8.REPORT COMPOSER

###8.1.OVERVIEW

**Cell 8 — Final report composition: board-facing memo structure, deterministic decision policy, and ‘Not verified’ discipline**

Cell 8 constructs `final_report.json`, the primary board-facing deliverable. This cell is where we translate the chain outputs into a memo that is readable, consistent, and governance-compliant. The final report is not free-form; it is a structured object with mandatory sections that align with how committees review decisions.

A major design feature here is the deterministic decision policy. Before relying on any LLM narrative, the notebook applies a simple, explicit rule set to computed metrics and the volume of open items. The rule set is deliberately conservative: multiple “stress signals” (for example, high leverage proxy, low EBITDA margin, low FCF proxy margin) push the recommendation to HUMAN_REVIEW, especially if open items are numerous. This is important because it ensures the system never uses LLM eloquence to override risk discipline. The LLM can help draft a rationale, but the governance posture is anchored in explicit policy and escalation rules.

Cell 8 also defines confidence in a structured way (low/medium/high with rationale). Confidence is not a “feeling”; it is tied to the completeness of inputs and the sensitivity of conclusions to missing information. If open items are material, confidence should be low and the recommendation should shift to HUMAN_REVIEW. This aligns with board expectations: the right answer is often “we need more information,” not “approve.”

The report explicitly separates:
- **facts_provided** (verbatim packet)
- **assumptions_introduced** (explicit list)
- **analysis** (computed metrics and assumption-based anchors)
- **recommendation** (GO/NO-GO/HUMAN_REVIEW)
- **open_items** and **questions_to_verify**
- **verification_status = Not verified**

This structure prevents “implicit verification.” It makes it impossible for the memo to appear final when it is not.

The cell also includes logic to incorporate LLM narrative only if it passes the unsupported-claims gate and is valid JSON. If the LLM fails, the memo still exists in deterministic form and escalates appropriately. That resilience is a production-grade characteristic: the system does not collapse if the model output is malformed.

In summary, Cell 8 is where governance becomes board-readable: it produces a consistent memo with explicit sections, explicit decision labels, explicit uncertainty, and explicit non-verification.


###8.2.CODE AND IMPLEMENTATION

In [13]:
# CELL 8/10 — Report composer (final_report.json) with strict schema
def decide_go_no_go(metrics: Dict[str, Any], open_items: List[str]) -> Tuple[str, str, List[str], Tuple[str, str]]:
    # Deterministic policy rules
    lev = metrics.get("leverage_proxy_debt_to_ebitda")
    ebitda_margin = float(metrics.get("ebitda_margin", 0.0))
    fcf_margin = float(metrics.get("fcf_proxy_margin", 0.0))

    caveats: List[str] = []
    rationale_parts: List[str] = []

    if lev is None:
        caveats.append("EBITDA is non-positive; leverage proxy not meaningful under current packet.")
        return "HUMAN_REVIEW", "EBITDA <= 0 prevents leverage-based screening.", caveats, ("low", "Key metric undefined under provided packet.")

    if lev >= CONFIG["decision_thresholds"]["leverage_proxy_high"]:
        caveats.append(f"Leverage proxy (Debt/EBITDA) is high at ~{lev:.2f}x; requires diligence on covenants and cash generation.")
    if ebitda_margin <= CONFIG["decision_thresholds"]["ebitda_margin_low"]:
        caveats.append(f"EBITDA margin is low at ~{ebitda_margin:.1%}; requires pricing/mix and cost validation.")
    if fcf_margin <= CONFIG["decision_thresholds"]["fcf_proxy_low"]:
        caveats.append(f"FCF proxy margin is low at ~{fcf_margin:.1%}; working capital and capex split are critical uncertainties.")

    # Decision rule: if any two stress signals -> HUMAN_REVIEW unless open items are minimal
    stress_signals = sum([
        lev >= CONFIG["decision_thresholds"]["leverage_proxy_high"],
        ebitda_margin <= CONFIG["decision_thresholds"]["ebitda_margin_low"],
        fcf_margin <= CONFIG["decision_thresholds"]["fcf_proxy_low"],
    ])

    if stress_signals >= 2:
        dec = "HUMAN_REVIEW"
        rationale_parts.append("Multiple financial stress signals under current packet; proceed only with tightened diligence scope and governance.")
        conf = ("low", "Packet is incomplete and stress signals suggest high sensitivity to missing items.")
    else:
        dec = "GO"
        rationale_parts.append("No more than one stress signal under current packet; proceeding to diligence is reasonable with clear open questions.")
        conf = ("medium", "Indicative view only; several open items could change the conclusion.")

    if len(open_items) >= 5:
        dec = "HUMAN_REVIEW"
        rationale_parts.append("Open items are numerous and material; governance requires human approval before any commitment.")
        conf = ("low", "Material unknowns remain unresolved.")

    return dec, " ".join(rationale_parts), caveats, conf

def compose_final_report(run_id: str, case: SyntheticCase, trace: Dict[str, Any], draft: Dict[str, Any], risks: List[Risk], gates: Dict[str, Any]) -> Dict[str, Any]:
    internal = trace.get("_internal", {})
    metrics = internal.get("metrics", {})
    anchors = internal.get("valuation_anchors", {})
    open_items = internal.get("open_items", [])
    assumptions_list = internal.get("assumptions_list", [])

    # Deterministic decision pre-gates
    dec, det_rationale, det_caveats, (conf_level, conf_rat) = decide_go_no_go(metrics, open_items)

    # Escalate decision if any gate failed or any high severity escalated risk
    gate_failed = not (gates["GateA_FactsVsAssumptions"].get("passed") and gates["GateB_UnsupportedClaims"].get("passed"))
    any_escalated = any(r.get("status") == "ESCALATED" and r.get("severity") == "high" for r in risks)
    if gate_failed or any_escalated:
        dec = "HUMAN_REVIEW"
        det_caveats = det_caveats + ["One or more governance gates failed; requires human review."]

    # Prefer LLM draft if valid JSON and no unsupported-claims gate failure
    exec_sum = ""
    rationale = det_rationale
    caveats = det_caveats
    questions = open_items[:]  # start from deterministic list
    if isinstance(draft, dict) and not draft.get("_parse_error") and gates["GateB_UnsupportedClaims"].get("passed"):
        exec_sum = str(draft.get("executive_summary","")).strip()
        if isinstance(draft.get("rationale"), str) and draft["rationale"].strip():
            rationale = draft["rationale"].strip()
        if isinstance(draft.get("caveats"), list) and all(isinstance(x,str) for x in draft["caveats"]):
            caveats = list(dict.fromkeys(caveats + draft["caveats"]))
        if isinstance(draft.get("questions_to_verify"), list) and all(isinstance(x,str) for x in draft["questions_to_verify"]):
            questions = list(dict.fromkeys(questions + draft["questions_to_verify"]))
        if draft.get("confidence_level") in ("low","medium","high"):
            conf_level = draft["confidence_level"]
        if isinstance(draft.get("confidence_rationale"), str) and draft["confidence_rationale"].strip():
            conf_rat = draft["confidence_rationale"].strip()

    if not exec_sum:
        exec_sum = (
            f"Indicative view (synthetic packet) for {case['company']}: "
            f"computed EBITDA and leverage/FCF proxies and constructed assumption-based valuation anchors. "
            f"Recommendation is {dec} pending resolution of material open items."
        )

    report = {
        "run_id": run_id,
        "timestamp_utc": utc_now(),
        "project": CONFIG["project"],
        "notebook": CONFIG["notebook"],
        "executive_summary": exec_sum,
        "facts_provided": case["facts_provided"],
        "assumptions_introduced": assumptions_list,
        "analysis": {
            "metrics": metrics,
            "valuation_anchors": anchors,
        },
        "recommendation": {
            "decision": dec,
            "rationale": rationale,
            "caveats": caveats,
        },
        "confidence": {
            "level": conf_level,
            "rationale": conf_rat,
        },
        "open_items": open_items,
        "questions_to_verify": questions,
        "verification_status": "Not verified",
        "_governance": {
            "gates": gates,
            "risk_summary": {
                "total": len(risks),
                "high": sum(1 for r in risks if r["severity"] == "high"),
                "escalated": sum(1 for r in risks if r["status"] == "ESCALATED"),
            }
        }
    }
    return report

print("REPORT COMPOSER READY")


REPORT COMPOSER READY


##9.ORCHESTRATION

###9.1.0VERVIEW

**Cell 9 — Orchestration, artifact writing, validation, and risk logging as an auditable run record**

Cell 9 is the run controller. It executes the pipeline end-to-end, writes all required artifacts, and ensures validation and escalation are actually enforced in practice. This is where governance stops being “a design intention” and becomes a repeatable operational procedure.

The run starts by creating a unique run_id and capturing a UTC timestamp. These identifiers are written into the run manifest, which records configuration, model choice, determinism settings, environment fingerprints, and output paths. For board oversight and audit readiness, the run manifest is the “cover sheet” of the workpaper package: it tells you what ran, when, and under what conditions.

Next, the cell calls `run_chain` to generate the reasoning trace, draft narrative, and preliminary risk list. Then it executes the gates:
- Gate A checks assumptions separation.
- Gate B checks for unsupported numeric claims and JSON parse compliance.
- The system composes a preliminary report.
- The trace is normalized and written.
- Gate C validates schemas for the trace and report.

This ordering matters. You cannot validate a report you have not composed, and you cannot finalize a report until you know whether the gates passed. The cell re-writes the trace after gate results are known to ensure the trace contains the final truth about the run.

The cell then enforces the schema one last time at write time. If schema validation fails, it logs a high-severity risk and forces the recommendation to HUMAN_REVIEW with low confidence. This is a critical safety property: even if something goes wrong, the system defaults to a conservative posture and records the failure.

Finally, the cell writes `risk_log.json` and `final_report.json`. The risk log is not an afterthought; it is the governance evidence that controls are functioning. It enables leadership to monitor how often the model fails constraints, what categories of failure occur, and where the system needs improvement.

From a pedagogical standpoint, Cell 9 teaches the “run discipline” of governed AI: every run generates artifacts, every artifact is validated, failures are logged, and outputs are packaged for review rather than casually printed to screen.


###9.2.CODE AND IMPLEMENTATION

In [14]:
# CELL 9/10 — Run orchestrator: executes pipeline end-to-end, writes artifacts
RUN_ID = "RUN-" + uuid.uuid4().hex[:12]
RUN_TS = utc_now()

# Run manifest (written early)
manifest = {
    "run_id": RUN_ID,
    "timestamp_utc": RUN_TS,
    "project": CONFIG["project"],
    "notebook": CONFIG["notebook"],
    "model": CONFIG["model"],
    "determinism": {"random_seed": 7, "PYTHONHASHSEED": os.environ.get("PYTHONHASHSEED","")},
    "environment": {"python": platform.python_version(), "platform": platform.platform()},
    "paths": {"artifacts": str(ART), "deliverables": str(DELIV)},
}
write_json(ART / "run_manifest.json", manifest)

# Execute chain
trace, draft, risks, _ = run_chain(CASE)

# Patch run_id into trace + prompts log linkage (we can't rewrite already logged lines; we record linkage in risk/governance)
# Update trace internal for gates
trace["_internal"]["case_facts"] = CASE["facts_provided"]

# Gate A
gA = gate_a_facts_vs_assumptions(trace, risks)
# Gate B
gB = gate_b_unsupported_claims(CASE, trace, risks)

# Compose report (pre Gate C)
gates = {
    "GateA_FactsVsAssumptions": gA,
    "GateB_UnsupportedClaims": gB,
    "GateC_SchemaValidity": {"passed": False},  # fill after validation
}
final_report = compose_final_report(RUN_ID, CASE, trace, draft, risks, gates)

# Normalize + write trace
trace_file_obj = build_reasoning_trace_file(trace, RUN_ID, gates)

# Gate C (schema validity) now that files are structured
gC = gate_c_schema_validity(final_report, trace_file_obj, risks)
gates["GateC_SchemaValidity"] = gC

# Re-write trace with final gate results
trace_file_obj = build_reasoning_trace_file(trace, RUN_ID, gates)

# Re-compose report with Gate C considered
final_report = compose_final_report(RUN_ID, CASE, trace, draft, risks, gates)

# Validate or force escalation
try:
    validate_or_raise(FINAL_REPORT_SCHEMA, final_report)
except Exception as e:
    risks.append({
        "risk_id": "R-" + uuid.uuid4().hex[:8],
        "timestamp_utc": utc_now(),
        "severity": "high",
        "category": "SCHEMA",
        "description": f"Final report schema invalid at write time: {str(e)[:600]}",
        "control": "Force HUMAN_REVIEW; do not use output for decisions.",
        "status": "ESCALATED",
    })
    # Force minimal safe report
    final_report["recommendation"]["decision"] = "HUMAN_REVIEW"
    final_report["confidence"]["level"] = "low"
    final_report["confidence"]["rationale"] = "Schema validation failed; output requires correction."

# Write risk log + final report
write_json(ART / "risk_log.json", {"run_id": RUN_ID, "timestamp_utc": utc_now(), "risks": risks})
write_json(ART / "final_report.json", final_report)

print("RUN COMPLETE:", {"run_id": RUN_ID, "decision": final_report["recommendation"]["decision"], "risks": len(risks)})


RUN COMPLETE: {'run_id': 'RUN-5dc5e5252314', 'decision': 'HUMAN_REVIEW', 'risks': 3}


##10.AUDIT BUNDLE

###10.1.OVERVIEW

**Cell 10 — Packaging, deliverables.zip, and operational handoff to reviewers**

Cell 10 completes the governance workflow by packaging the run outputs into a single deliverable archive: `deliverables/deliverables.zip`. This might appear administrative, but it is central to making the notebook usable in real professional environments. Boards and review committees do not want to chase files across a notebook runtime. They want a clean handoff package that can be stored, shared internally, and reviewed consistently.

The cell creates a zip file that includes all artifacts under an `artifacts/` directory and a convenient copy of the final report at the root of the zip. This dual structure supports two review modes:
- A quick review mode where a reviewer opens the final report first.
- A deeper audit mode where a reviewer inspects run_manifest, reasoning_trace, prompts_log, and risk_log.

Packaging also supports retention and versioning. In organizations with strong governance, analysis outputs are retained as evidence, often in controlled repositories. A single zip bundle can be hashed, archived, and referenced in meeting materials. This is especially important for AI systems: you want to preserve “what the system said at the time,” not rely on re-running later under different conditions.

The cell also prints a minimal console summary showing the run_id, timestamp, and file paths. This is an operational usability feature: it reduces friction for analysts and reviewers and encourages correct handling of outputs. In production, this could be replaced by a dashboard, a document management push, or an automated routing to reviewers, but the principle remains: the system should end with a clear handoff artifact, not an ambiguous state.

Pedagogically, Cell 10 teaches that governance is not only about controls inside the model call. Governance also includes distribution discipline: how outputs are packaged, stored, and delivered for review. A pipeline that produces excellent analysis but cannot produce a clean review bundle will fail in institutional settings because it cannot integrate into existing oversight workflows.

In short, Cell 10 turns the run into a board-ready package: one zip file, consistent contents, and a clear operational endpoint.


###10.2.CODE AND IMPLEMENTATION

In [15]:
# CELL 10/10 — Packaging: zip deliverables + minimal console summary of outputs/paths
zip_path = DELIV / "deliverables.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ART.glob("**/*")):
        if p.is_file():
            z.write(p, arcname=str(Path("artifacts") / p.relative_to(ART)))
    # Include a copy of final_report at root for convenience
    z.write(ART / "final_report.json", arcname="final_report.json")

# Minimal console summary (paths only)
outputs = {
    "artifacts/run_manifest.json": str(ART / "run_manifest.json"),
    "artifacts/prompts_log.jsonl": str(ART / "prompts_log.jsonl"),
    "artifacts/reasoning_trace.json": str(ART / "reasoning_trace.json"),
    "artifacts/risk_log.json": str(ART / "risk_log.json"),
    "artifacts/final_report.json": str(ART / "final_report.json"),
    "deliverables/deliverables.zip": str(zip_path),
}
print(json.dumps({"run_id": RUN_ID, "timestamp_utc": utc_now(), "outputs": outputs}, indent=2))


{
  "run_id": "RUN-5dc5e5252314",
  "timestamp_utc": "2026-02-19T21:29:52.929229+00:00",
  "outputs": {
    "artifacts/run_manifest.json": "/content/artifacts/run_manifest.json",
    "artifacts/prompts_log.jsonl": "/content/artifacts/prompts_log.jsonl",
    "artifacts/reasoning_trace.json": "/content/artifacts/reasoning_trace.json",
    "artifacts/risk_log.json": "/content/artifacts/risk_log.json",
    "artifacts/final_report.json": "/content/artifacts/final_report.json",
    "deliverables/deliverables.zip": "/content/deliverables/deliverables.zip"
  }
}


In [16]:
# Paste this into a Colab cell to print a board-readable report from artifacts/final_report.json
import json
from pathlib import Path

p = Path("artifacts/final_report.json")
if not p.exists():
    raise FileNotFoundError("artifacts/final_report.json not found. Run the notebook pipeline first (Cell 9).")

r = json.loads(p.read_text(encoding="utf-8"))

def _money(x):
    try:
        return f"${float(x):,.0f}"
    except Exception:
        return str(x)

def _pct(x):
    try:
        return f"{float(x)*100:.1f}%"
    except Exception:
        return str(x)

def _fmt(x):
    if isinstance(x, (int, float)):
        return _money(x)
    return str(x)

company = r.get("facts_provided", {}).get("deal_context", {}).get("company_name", None)
company = company or "MidCo Industrial Services, Inc. (synthetic)"

facts = r.get("facts_provided", {})
is_ = facts.get("income_statement", {})
bs_ = facts.get("balance_sheet_highlights", {})
kpis = facts.get("operating_kpis", {})
analysis = r.get("analysis", {})
metrics = analysis.get("metrics", {})
anchors = analysis.get("valuation_anchors", {})

decision = r.get("recommendation", {}).get("decision", "HUMAN_REVIEW")
rec_rationale = r.get("recommendation", {}).get("rationale", "")
caveats = r.get("recommendation", {}).get("caveats", [])
conf = r.get("confidence", {})
conf_level = conf.get("level", "low")
conf_rat = conf.get("rationale", "")

open_items = r.get("open_items", [])
questions = r.get("questions_to_verify", [])
assumptions = r.get("assumptions_introduced", [])
exec_sum = r.get("executive_summary", "")
run_id = r.get("run_id", "")
ts = r.get("timestamp_utc", "")
verification = r.get("verification_status", "Not verified")

# Metrics display
rev = metrics.get("revenue", is_.get("revenue"))
ebitda = metrics.get("ebitda")
ebitda_m = metrics.get("ebitda_margin")
lev = metrics.get("leverage_proxy_debt_to_ebitda")
fcf = metrics.get("fcf_proxy")
fcf_m = metrics.get("fcf_proxy_margin")
debt = bs_.get("total_debt")

# Valuation anchors display
mults = anchors.get("multiples_assumed", {})
evr = anchors.get("enterprise_value_range", {})

line = "=" * 86
print(line)
print("BOARD BRIEF — DETERMINISTIC CHAIN REASONING (GOVERNANCE-FIRST)")
print(line)
print(f"Run ID: {run_id}")
print(f"Timestamp (UTC): {ts}")
print(f"Verification Status: {verification}")
print()

print("PROBLEM WE ARE SOLVING")
print("-" * 86)
print("We need a disciplined, auditable way to answer an early-stage deal question using a bounded input")
print("packet, while preventing invented facts and forcing explicit separation of facts vs assumptions.")
print()
print("DEAL QUESTION")
print("-" * 86)
print("Should we proceed to diligence at indicative valuation?")
print()

print("EXECUTIVE SUMMARY")
print("-" * 86)
print(exec_sum or "(No executive summary found in final_report.json)")
print()

print("KEY FACTS PROVIDED (BOUNDED PACKET)")
print("-" * 86)
print(f"Period: {facts.get('period','(missing)')}  |  Currency: {facts.get('currency','(missing)')}")
print(f"Revenue: {_money(is_.get('revenue','(missing)'))}")
print(f"COGS: {_money(is_.get('cogs','(missing)'))}  |  Opex: {_money(is_.get('opex','(missing)'))}  |  D&A: {_money(is_.get('d_and_a','(missing)'))}")
print(f"Interest Expense: {_money(is_.get('interest_expense','(missing)'))}")
print(f"Total Debt: {_money(debt if debt is not None else '(missing)')}  |  Cash: {_money(bs_.get('cash','(missing)'))}")
if kpis:
    print("Operating KPIs:")
    for k, v in kpis.items():
        if isinstance(v, float) and v <= 1.0:
            print(f"  - {k}: {_pct(v)}")
        else:
            print(f"  - {k}: {v}")
print()

print("ASSUMPTIONS INTRODUCED (EXPLICIT)")
print("-" * 86)
if assumptions:
    for a in assumptions:
        print(f"  - {a}")
else:
    print("  (none listed)")
print()

print("DERIVED METRICS (COMPUTED)")
print("-" * 86)
print(f"EBITDA: {_money(ebitda)}  |  EBITDA Margin: {_pct(ebitda_m)}")
print(f"Leverage Proxy (Debt/EBITDA): {('n/a' if lev is None else f'{lev:.2f}x')}")
print(f"FCF Proxy: {_money(fcf)}  |  FCF Proxy Margin: {_pct(fcf_m)}")
print()

print("INDICATIVE VALUATION ANCHORS (ASSUMPTION-BASED, NOT MARKET COMPS)")
print("-" * 86)
if evr and mults:
    print(f"Assumed EV/EBITDA Multiples: low={mults.get('low')}x  mid={mults.get('mid')}x  high={mults.get('high')}x")
    print(f"Enterprise Value Range: low={_money(evr.get('low'))}  mid={_money(evr.get('mid'))}  high={_money(evr.get('high'))}")
else:
    print(json.dumps(anchors, indent=2))
print()

print("RECOMMENDATION (CONTROL-GRADE)")
print("-" * 86)
print(f"Decision: {decision}")
print(f"Confidence: {conf_level}")
if conf_rat:
    print(f"Confidence Rationale: {conf_rat}")
print()
print("Rationale:")
print(rec_rationale or "(missing)")
print()
print("Caveats:")
if caveats:
    for c in caveats:
        print(f"  - {c}")
else:
    print("  (none)")
print()

print("OPEN ITEMS / QUESTIONS TO VERIFY (WHAT MUST BE CONFIRMED IN DILIGENCE)")
print("-" * 86)
merged = []
seen = set()
for x in (open_items or []) + (questions or []):
    if isinstance(x, str) and x.strip() and x.strip() not in seen:
        seen.add(x.strip())
        merged.append(x.strip())
if merged:
    for q in merged:
        print(f"  - {q}")
else:
    print("  (none)")
print()

print("GOVERNANCE STATUS")
print("-" * 86)
gov = r.get("_governance", {})
gates = gov.get("gates", {})
if gates:
    for k, v in gates.items():
        passed = v.get("passed", False) if isinstance(v, dict) else False
        print(f"{k}: {'PASSED' if passed else 'FAILED'}")
else:
    print("(gate results missing in report)")
print()
print(line)


BOARD BRIEF — DETERMINISTIC CHAIN REASONING (GOVERNANCE-FIRST)
Run ID: RUN-5dc5e5252314
Timestamp (UTC): 2026-02-19T21:29:36.702326+00:00
Verification Status: Not verified

PROBLEM WE ARE SOLVING
--------------------------------------------------------------------------------------
We need a disciplined, auditable way to answer an early-stage deal question using a bounded input
packet, while preventing invented facts and forcing explicit separation of facts vs assumptions.

DEAL QUESTION
--------------------------------------------------------------------------------------
Should we proceed to diligence at indicative valuation?

EXECUTIVE SUMMARY
--------------------------------------------------------------------------------------
Indicative view (synthetic packet) for MidCo Industrial Services, Inc. (synthetic): computed EBITDA and leverage/FCF proxies and constructed assumption-based valuation anchors. Recommendation is HUMAN_REVIEW pending resolution of material open items.

KEY FA

##11.CONCLUSION

**Conclusion — What Notebook 1 Achieved, What It Cannot Do Yet, and How We Build Forward**

This first notebook is a deliberately conservative starting point. It proves that we can take a finance question that normally invites informal judgment and “hand-wavy” narrative, and instead run it through a controlled reasoning pipeline that behaves like a disciplined internal process: bounded inputs, explicit steps, explicit artifacts, and explicit escalation when controls fail. The contribution is not the specific metrics computed or the final recommendation label. The contribution is the institutional posture: **mechanism over mystique**. We are not asking anyone to trust a model’s eloquence; we are demonstrating a governed workflow that produces reviewable workpapers.

The most valuable positive contribution is the **conversion of reasoning into an artifact**. In many organizations, the “analysis” exists in a human’s head and the memo is a thin veneer over a complex mental process. That is fragile: it is hard to audit, hard to reproduce, and difficult to challenge constructively. Notebook 1 forces the analysis into a structured chain of steps, each of which is inspectable. That is a fundamental governance upgrade. A reviewer can see, step by step, what facts were used, what assumptions were introduced, what values were derived, what uncertainties remain, and why the process moved forward. This makes disagreement productive. Instead of debating vibes, we can debate the step where an assumption was introduced, or the sensitivity of a valuation anchor, or the completeness of the open-items list.

The second positive contribution is **hard separation of facts and assumptions** as an enforceable control, not a stylistic preference. Finance decisions break down when assumptions are implicit—especially when they are embedded inside prose. Notebook 1 treats assumptions as first-class objects: they are listed, preserved, and carried into the final report as their own section. That structure is board-appropriate because it aligns with how committees actually operate: they do not approve “analysis,” they approve an analysis conditional on assumptions being reasonable and verified.

Third, the notebook proves a practical approach to AI risk: **the system is designed to stop**. The unsupported claim detector (Gate B) and schema enforcement (Gate C) are not academic. They operationalize the idea that the model is a component that must earn trust each run. If the LLM introduces unsupported numeric claims or fails to comply with the required structure, the pipeline escalates to HUMAN_REVIEW and logs the failure in a control-grade risk register. In other words, the system treats hallucination and non-compliance as exceptions, not as “quirks” that humans must mentally filter. That shift is what makes AI usable in professional environments without drifting into liability.

Fourth, the governance bundle is itself a meaningful deliverable. The run manifest, prompts log (redacted and hashed), reasoning trace, risk log, final report, and zip packaging collectively create a repeatable record. That record matters for oversight: it allows review sampling, incident analysis, post-mortems, training, and future automation. Even if the memo is not used for a real decision, the artifacts demonstrate an institutional pattern that can be scaled.

At the same time, Notebook 1 is intentionally limited, and those limitations are important to state clearly, because boards should not confuse “structured” with “complete.” This notebook **does not** do market work. It does not bring external data, real comparable multiples, industry benchmarks, or live pricing. Its valuation anchors are deliberately assumption-based placeholders, useful only for illustrating how to structure indicative ranges without smuggling in unverified “facts.” In real screening, you would want controlled sources, documented provenance, and review steps for any external comps or market references. Notebook 1 is the governance scaffold, not the full research product.

It also cannot yet guarantee semantic correctness of qualitative claims. While Gate B is strict on numeric claims, qualitative statements can still drift if not tightly constrained. The notebook addresses this partially by enforcing a bounded packet and instructing the LLM to avoid external facts, but this is not a complete solution. A future improvement is a stronger “claim grounding” control: every assertion—numeric or qualitative—should be mapped to either (a) a fact key in the packet, (b) an explicit assumption entry, or (c) an open item. That is the natural next step if we want higher reliability in narrative sections.

Another limitation is that the notebook is not yet a full committee workflow. It produces a board-facing report, but it does not implement multi-reviewer approvals, sign-off capture, redline loops, or formal routing. It can escalate to HUMAN_REVIEW, but it cannot yet “hand the baton” to named roles with tracked decisions and accountability. In production, escalation is only meaningful if there is an actual process behind it: who reviews, what they check, what they approve, and what gets recorded. Notebook 1 demonstrates the trigger; future notebooks will demonstrate richer routing and governance patterns.

It also does not yet implement robust sensitivity analysis or scenario exploration. Chain reasoning is intentionally single-path: it is good for producing a consistent memo structure, but it does not help the board compare strategic alternatives, test different scenarios, or evaluate competing theses. That is not a flaw; it is a design choice. Single-path reasoning is the foundation because it creates discipline around bounded inputs, structured outputs, and gating. Once that foundation is proven, we can responsibly add branching and iteration without losing control.

Finally, while we strive for determinism (fixed seeds, temperature 0, pinned dependencies), we should be explicit that deterministic settings reduce variance but do not eliminate it entirely in hosted model APIs. This is why the artifacts and hashes matter: they provide practical reproducibility, where we can detect drift and audit changes even if perfect determinism is not possible.

These improvement areas are not “nice to have.” They define the path from a strong prototype to a production-grade capability:

- Strengthen claim-grounding so qualitative statements must cite fact keys, assumptions, or open items.
- Add structured human review workflows: named approvers, approval metadata, and sign-off artifacts.
- Extend controls for external data ingestion with provenance, caching, and documented source governance.
- Add explicit sensitivity analysis and scenario comparison in controlled forms.
- Improve evaluation: build deterministic checks that score outputs on assumption leakage, uncertainty disclosure, and policy compliance.

This sets up the bridge to future chapters and notebooks. The five-notebook sequence is intentionally architectural. Notebook 1 (Chain) establishes the minimal contract: bounded inputs, explicit reasoning trace, strict separation of facts and assumptions, and enforced escalation. With that baseline in place, we can expand reasoning capacity without sacrificing governance:

- The next step is **Tree Reasoning**. If Notebook 1 answers “what do we think, given a single thesis,” Notebook 2 will answer “what do we think across multiple strategic paths.” Tree reasoning introduces controlled branching (Acquire vs Divest vs Do Nothing), explicit pruning rules, and audit evidence for why alternatives were discarded. This is board-relevant because it mirrors how strategic committees actually decide: not by a single narrative, but by comparing structured options under constraints.

- After trees, we move to **Loop Reasoning**. In real credit and diligence work, the memo is refined iteratively as questions are asked and risks are discovered. Notebook 3 will formalize iteration, convergence criteria, and termination rules—so the system can improve a memo over multiple cycles without running uncontrolled, and can stop decisively when material unknowns remain. This is where governance meets operations: the system learns to say “we cannot proceed without these answers,” and it can prove why.

- Then we reach **Committee Reasoning**. Boards and ICs are not monolithic. They depend on roles, dissent, veto policies, and quorum. Notebook 4 will simulate a four-role committee, preserve dissent in structured form, and enforce policy rules (for example, a compliance veto). This will transform “one memo” into a controlled debate with recorded objections—exactly the kind of structure that improves accountability in sensitive decisions.

- Finally, we introduce **Trainable Reasoning** as measured improvement, not mythology. Notebook 5 will compare a baseline prompt to an adapted governance-first prompt, evaluate both outputs with deterministic metrics, and produce an evidence-based recommendation for deployment constraints. This is how we move from “we think this prompt is better” to “we can prove it reduces assumption leakage and improves uncertainty disclosure under a fixed rubric.”

In summary, Notebook 1 is a strong starting point precisely because it is strict. It shows that we can build AI-assisted finance memos that are not just readable, but governable. It makes the reasoning process visible, it makes the boundary between facts and assumptions explicit, it creates hard stop conditions when the model violates policy, and it produces a bundle of artifacts that a reviewer can inspect. It cannot yet do external market work, it cannot yet fully ground qualitative claims, and it does not yet implement multi-role workflows or scenario exploration—but those are not missing pieces in the same system; they are the next architectural layers in a planned progression. This notebook gives the board something rare in AI: a controlled foundation on which we can safely add complexity.
